이번 실습은 아마존 상품 정보 데이터를 추출해서 사용자에게 적합한 정보를 전달하는 요약문을 만들어 볼겁니다.

오픈소스 GPT를 이용하여 데이터 생성 파이프라인이 어떤것이고 어떻게 요약문을 만들어가는지 살펴보려 합니다.



In [3]:
# vLLM 은 별도 venv 에서 서버로 띄운다 (verify/02_vllm_venv.sh 참조).
%pip install -q -U openai datasets pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 24.1 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
  Attempting uninstall: datasets
    Found existing installation: datasets 2.14.4
    Uninstalling datasets-2.14.4:
      Successfully uninstalled datasets-2.14.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 25.2.1 requires numba<0.61.0a0,>=0.59.1, but you have numba 0.61.2 which is incompatible.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.


In [1]:
import json
from datasets import load_dataset
from openai import OpenAI

from pydantic import BaseModel
from typing import List

먼저 사용할 아마존 데이터를 불러옵니다.

In [2]:
data = load_dataset('Taekyoon/test_amazon', split='train[:10]')

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


amazon_ec_examples.json:   0%|          | 0.00/190k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/100 [00:00<?, ? examples/s]

In [4]:
data

Dataset({
    features: ['text'],
    num_rows: 10
})

In [3]:
print(data[0]['text'])


### Product Catalog Information

#### Product Information

- Product Title: FS-1051 FATSHARK TELEPORTER V3 HEADSET
- Brand: F, a, t,  , S, h, a, r, k
- Categories: Electronics > Television & Video > Video Glasses
- Product Attributes: 
Date First Available: August 2, 2014
Manufacturer: Fatshark

- Product Description: 

Description:
Teleporter V3 The “Teleporter V3” kit sets a new level of value in the FPV world with Fat Shark renowned performance and quality. The fun of FPV is experienced firsthand through the large screen FPV headset with integrated NexwaveRF receiver technology while simultaneously recording onboard HD footage with the included “PilotHD” camera. The “Teleporter V3” kit comes complete with everything you need to step into the cockpit of your FPV vehicle. We’ve included our powerful 250mW 5.8Ghz transmitter, 25 degree FOV headset (largest QVGA display available), the brand new “PilotHD” camera with live AV out and all the cables, antennas and connectors needed.



여러분들이 실습한 GPT 모델을 실행해보겠습니다.
실행후에 강사님의 지시가 있을 때 까지 다른 코드 실행을 하지 말아주세요.

In [5]:
# 별도 터미널에서 실행한다:
#   source /opt/vllm-env/bin/activate
#   nohup vllm serve Qwen/Qwen3-4B-Instruct-2507 --port 8000 > vllm.log 2>&1 &
!curl -s http://localhost:8000/v1/models || echo "서버가 아직 준비되지 않았습니다"

nohup: appending output to 'nohup.out'


In [ ]:
# 모델이 잘 실행되는지 테스트 해봅니다.
openai_api_key = "EMPTY"
openai_api_base = "http://localhost:8000/v1"
model_name = "Qwen/Qwen3-4B-Instruct-2507"

client = OpenAI(
    api_key=openai_api_key,
    base_url=openai_api_base,
)
completion = client.completions.create(model=model_name,
                                      prompt="정보 이론에 대해서 설명해주세요.")
print("Completion result:", completion.choices[0].text)

### Step 1

모델이 이해하는 제품 정보가 무엇인지 알아보기

- 데이터 추출 시에는 내가 요구하는것을 모델에 요청하는 것보다 모델이 무엇을 알고있는지를 물어보는게 문제해결에 도움이 됩니다.

In [ ]:
STEP_1_PROMPT = """Objective:
Analyze the provided product description and identify the types of features that can be attributed to the product.

Task:
1. List the names of the feature types (e.g., color, size, material, etc.).
2. For each feature type, provide a brief description explaining what it represents.

Key Considerations:

1. Explicit Mention: Only include feature types that are explicitly mentioned in the product description to ensure accuracy.
2. Clear Descriptions: Ensure each description is concise and clearly explains the feature type without including specific values.
3. JSON Validity: Verify the JSON format for proper syntax to ensure usability and avoid errors.

Example of Desired Output:

[
    {"feature_type": "Color", "descript": "The available shades or hues of the product."},
    {"feature_type": "Size", "descript": "The dimensions or measurements of the product."},
    {"feature_type": "Material", "descript": "The substance used to make the product."},
    {"feature_type": "Brand", "descript": "The manufacturer or brand name of the product."},
    {"feature_type": "Release Year", "descript": "The year the product was released."},
]
"""

In [ ]:
class FeatureType(BaseModel):
    feature_type: str
    descript: str

class FeatureTypeList(BaseModel):
    feature_type_list: List[FeatureType]


feature_type_schema = FeatureTypeList.model_json_schema()


def gen_text_step_1(text):
    try:
        messages = []

        messages.append({"role": "user", "content": STEP_1_PROMPT + '\n\n' + text})

        completion = client.chat.completions.create(
              model=model_name,
              messages=messages,
              temperature=0.8,
              top_p=0.95,
              response_format={"type": "json_schema", "json_schema":
                  {"name": "feature_type_list", "schema": feature_type_schema}},
            )

        return completion.choices[0].message.content
    except Exception as e:
        print(e)
        return 'error'

In [ ]:
feature_type_list = gen_text_step_1(data[0]['text'])
print(feature_type_list)

### Step 2

모델이 이해하는 제품 정보를 묶어보기

- 모델이 알고있는 정보가 많을 경우에는 정보를 그룹으로 만들어서 사람 입장에서도 보기 쉬운 데이터 구성을 해둡니다.

In [ ]:
STEP_2_PROMPT = """Objective:
Categorize the identified features into the following subsections

Task:

- Organize into Subsections: Group the identified feature types into the predefined subsections.
- JSON Output: Present the final result in a structured JSON format, with each subsection containing an array of feature objects.

Key Considerations:

- Explicit Mention: Only include feature types that are explicitly mentioned in the product description to ensure accuracy.
- Clear Descriptions: Ensure each description is concise and clearly explains the feature type without including specific values.
- JSON Validity: Verify the JSON format for proper syntax to ensure usability and avoid errors.

Example of Desired Output:

[
    {
        "subsection": "Product Specifications",
        "features": ["Color", "Size", "Material"]
    },
    {
        "subsection": "Product Identification",
        "features": ["Brand", "Release Year"]
    }
]
"""

In [ ]:
class Subsection(BaseModel):
    subsection: str
    features: List[str]

class SubsectionList(BaseModel):
    subsection_list: List[Subsection]


subsectoin_schema = SubsectionList.model_json_schema()


def gen_text_step_2(text):
    try:
        messages = []

        messages.append({"role": "user", "content": STEP_2_PROMPT + '\n\n' + text})

        completion = client.chat.completions.create(
              model=model_name,
              messages=messages,
              max_tokens=1024,
              top_p=0.95,
              seed=777,
              response_format={"type": "json_schema", "json_schema":
                  {"name": "subsection_list", "schema": subsectoin_schema}},
            )

        return completion.choices[0].message.content
    except Exception as e:
        print(e)
        return 'error'

In [ ]:
subsection_list = gen_text_step_2(feature_type_list)
print(subsection_list)

### Step 3

제품 정보 추출하기

- 정보를 추출할 때는 내가 추출할 대상을 명시적으로 언급해주는 것이 도움이 됩니다.

In [ ]:
STEP_3_PROMPT = """Objective:
Extract and list details from the source text using the provided features.

Task:
1. Extract relevant details from the source text.
2. List the extracted details using the provided JSON formats.

Key Considerations:
1. Ensure the extracted details are accurate and complete.
2. Organize the listed details clearly and properly using the specified JSON structure.

Example of Desired Output:

[
    {"feature_type": "Color", "value": "yellow"},
    {"feature_type": "Brand", "value": "APPLE"},
    {"feature_type": "Release Year", "value": "2000"},
]
"""

In [ ]:
class ExtractedFeature(BaseModel):
    feature_type: str
    value: str

class ExtractedFeatureList(BaseModel):
    feature_list: List[ExtractedFeature]


extracted_feature_schema = ExtractedFeatureList.model_json_schema()


def gen_text_step_3(text):
    try:
        messages = []

        messages.append({"role": "user", "content": STEP_3_PROMPT + '\n\n' + text})

        completion = client.chat.completions.create(
              model=model_name,
              messages=messages,
              max_tokens=1024,
              top_p=0.95,
              seed=0,
              response_format={"type": "json_schema", "json_schema":
                  {"name": "extracted_features", "schema": extracted_feature_schema}},
            )

        return completion.choices[0].message.content
    except Exception as e:
        print(e)
        return 'error'

앞서 Step 1에서 추출한 내용을 Prompt에 같이 넣어볼까요?

In [ ]:
print(feature_type_list)

In [ ]:
extracted_feature_list = gen_text_step_3("Provided Features:\n\n"+ feature_type_list + "\n\n" + data[0]['text'])
features = json.loads(extracted_feature_list)
features

### Step 4

제품을 기대하는 고객에 대한 인사이트 물어보기

- 때로는 GPT 모델이 생성한 인사이트 결과를 실험적으로 활용하기에 용이할 때가 있습니다.
- 이번 실습에서는 사람이 직접 인사이트를 분석할 시간은 대신해서 GPT에게 물어봅시다.

In [ ]:
STEP_4_PROMPT = """Objective:
Categorize potential buyers of this product based on their intended use or purposes.
Limit 3 to 4 Categories

Task:
1. Identify distinct categories of users who are likely to purchase this product.
2. List the user categories using the provided JSON format.

Key Considerations:
1. Use common sense and logical reasoning to group users into meaningful categories.
2. Ensure the category explanations are clear and concise.
3. Organize the listed categories in a clear and proper JSON structure, as shown below:

[
  {"user_category": "category name", "describe": "explanation of this category"},
  {"user_category": "category name", "describe": "explanation of this category"},
  ...
]
"""

In [ ]:
class ConsumerCategory(BaseModel):
    user_category: str
    describe: str

class ConsumerCategoryList(BaseModel):
    consuber_categories: List[ConsumerCategory]

consumer_category_schema = ConsumerCategoryList.model_json_schema()

def gen_text_step_4(text):
    try:
        messages = []
        messages.append({"role": "user", "content": STEP_4_PROMPT + '\n\n' + text})

        completion = client.chat.completions.create(
              model=model_name, messages=messages,
              max_tokens=1024, top_p=0.95, seed=1234,
              response_format={"type": "json_schema", "json_schema":
                  {"name": "consumer_categories", "schema": consumer_category_schema}},
            )

        return completion.choices[0].message.content
    except Exception as e:
        print(e)
        return 'error'

In [ ]:
user_category_list = gen_text_step_4(data[0]['text'])
print(user_category_list)

### Step 5

요약문을 만들기 위해 추출한 정보 정리하기

- 요약문을 만들기 위해 앞서 추출한 데이터를 Subsection별로 묶어보겠습니다.

In [ ]:
# Feature list를 dict 형태로 바꾸기
feature_map = {row['feature_type']: row['value']
               for row in features['feature_list']}

In [ ]:
feature_map

In [ ]:
# 각 Feature를 Subsection에 맞게 정리하기
subsection_dict = {}

subsection_list = json.loads(subsection_list)

for row in subsection_list['subsection_list']:
    subsection = row['subsection']

    subsection_dict[subsection] = {f: feature_map[f]
                                   for f in row['features']
                                   if f in feature_map}

In [ ]:
subsection_dict

### Step 6

제품 정보만 모아 요약하기

- 사용자에게 전달할 문장을 만들기 전에 제품 정보만 미리 문장으로 구성해 보겠습니다.

In [ ]:
FACTUAL_SUMMARY_PROMPT = """Given Json Format inputs, condense the product information into a concise, factual one line text. Please follow these example cases below.

{"summary": "<Short Feature Express>: <Detailed One line Summary>"}

Please output as a json format"""

In [ ]:
class SummaryDescription(BaseModel):
    summary: str

summary_schema = SummaryDescription.model_json_schema()

def gen_text_summary(text):
    try:
        messages = []
        messages.append({"role": "user", "content": FACTUAL_SUMMARY_PROMPT + '\n\n' + text})

        completion = client.chat.completions.create(
              model=model_name, messages=messages,
              max_tokens=1024, top_p=0.95, seed=1234,
              response_format={"type": "json_schema", "json_schema":
                  {"name": "summary", "schema": summary_schema}},
            )

        return completion.choices[0].message.content
    except Exception as e:
        print(e)
        return 'error'

In [ ]:
# dict 객체로 된 Subsection을 다시 텍스트로 표현합니다.
subsection_list = []

for k, v in subsection_dict.items():
    subsection_list.append((k, json.dumps(v, indent=4)))

In [ ]:
subsection_list

In [ ]:
subsection_summary = {}

for row in subsection_list:
    summary_text = gen_text_summary(row[0] + '\n\n' + row[1])
    subsection_summary[row[0]] = summary_text

In [ ]:
subsection_summary

### Step 7

고객에 대한 특징을 첨가하여 요약하기

- 구매할 사용자에게 전달할 내용을 앞서 만든 제품 정보 요약과 합쳐보겠습니다.

In [ ]:
FEATURED_SUMMARY_PROMPT = """Given inputs, make multiple lines of feature summaries in English only and provide only the summarized sentence as the output.

Please output as a json format"""

In [ ]:
class SummaryDescription(BaseModel):
    summary: str

summary_schema = SummaryDescription.model_json_schema()

def gen_text_featured_summary(text):
    try:
        messages = []

        messages.append({"role": "user", "content": FEATURED_SUMMARY_PROMPT + '\n\n' + text})

        completion = client.chat.completions.create(
              model=model_name, messages=messages,
              max_tokens=1024, top_p=0.95, seed=1234,
              response_format={"type": "json_schema", "json_schema":
                  {"name": "summary", "schema": summary_schema}},
            )

        return completion.choices[0].message.content
    except Exception as e:
        print(e)
        return 'error'

In [ ]:
# user category list를 dict로 변환
describe_by_consumer = {row['user_category']: row['describe']
                        for row in json.loads(user_category_list)['consuber_categories']}

In [ ]:
describe_by_consumer

In [ ]:
# 제품 정보 요약 문장들을 하나로 묶기
feature_summaries = '\n'.join([json.loads(v)['summary'] for k, v in subsection_summary.items()])
print(feature_summaries)

In [ ]:
total_summary_by_consumer = {}

for k, v in describe_by_consumer.items():
    content = f"Consumer Type: {k}; Consumer Objectives: {v}; Related Feature Summaries: {feature_summaries}"
    total_summary_by_consumer[k] = gen_text_featured_summary(content)

In [ ]:
total_summary_by_consumer

### Step Final

지금까지 만들어온 과정을 하나의 파이프라인으로 만들어보겠습니다.

- datasets 라이브러리를 활용하여 데이터 생성 파이프라인을 만들어 봅시다.

In [ ]:
# Step 1
data = data.map(lambda x: dict(feature_type_list_str=gen_text_step_1(x['text'])))
# Step 2
data = data.map(lambda x: dict(subsection_list_str=gen_text_step_2(x['feature_type_list_str'])))
# Step 3
data = data.map(lambda x: dict(extracted_feature_list_str=
                               gen_text_step_3("Provided Features:\n\n"+ x['feature_type_list_str'] + "\n\n" + x['text'])))
# Step 4
data = data.map(lambda x: dict(user_category_list_str=gen_text_step_4(x['text'])))

In [ ]:
# Step 5

def organize_features(x):
    subsection_dict = {}
    feature_map = json.loads(x['feature_map'])

    subsection_list = json.loads(x['subsection_list_str'])

    for row in subsection_list['subsection_list']:
        subsection = row['subsection']

        subsection_dict[subsection] = {f: feature_map[f]
                                      for f in row['features']
                                      if f in feature_map}

    return dict(subsection_features=json.dumps(subsection_dict))

data = data.map(lambda x: dict(feature_map=json.dumps({row['feature_type']: row['value']
                                                      for row in json.loads(x['extracted_feature_list_str'])['feature_list']})))
data = data.map(organize_features)

In [ ]:
# Step 6

def summarize_subsection(x):
    summary_by_subsection = {}
    subsection_features = json.loads(x['subsection_features'])

    for k, v in subsection_features.items():
        name, features = k, json.dumps(v, indent=4)
        summary_text = gen_text_summary(name + '\n\n' + features)
        summary_by_subsection[name] = summary_text

    return dict(summary_by_subsection=json.dumps(summary_by_subsection))

data = data.map(summarize_subsection)

In [ ]:
def summarize_by_users(x):
    user_category_list = json.loads(x['user_category_list_str'])
    summary_by_subsection = json.loads(x['summary_by_subsection'])

    describe_by_consumer = {row['user_category']: row['describe']
                            for row in user_category_list['consuber_categories']}

    feature_summaries = '\n'.join([json.loads(v)['summary']
                                   for k, v in summary_by_subsection.items()])

    total_summary_by_consumer = {}

    for k, v in describe_by_consumer.items():
        content = f"Consumer Type: {k}; Consumer Objectives: {v}; Feature Summaries: {feature_summaries}"
        total_summary_by_consumer[k] = gen_text_featured_summary(content)

    return dict(summary_by_users=json.dumps(total_summary_by_consumer))

data = data.map(summarize_by_users)

In [ ]:
output_data = data.select_columns(['text', 'summary_by_users', 'summary_by_subsection'])

In [ ]:
output_data.to_csv('amazon_gpt_summary.csv')